# 09.05 -- LLM Serving: Production Deployment

**Module:** 09 -- LLM Inference Engineering  
**Notebook:** 5 of 5  
**Prerequisites:** 09.01 through 09.04

---

## Learning Objectives

1. Understand the full production LLM serving stack and how the components fit together
2. Set up and benchmark a local vLLM server with realistic load
3. Implement a load-testing harness that measures TTFT, TPOT, and throughput
4. Understand the capacity planning formula: how many GPUs do you need?
5. Apply the production deployment checklist: observability, rate limiting, and health checks

---

## 1. The Production LLM Serving Stack

A production LLM serving system has several layers, each with distinct engineering concerns:

```
Client (browser / app / API consumer)
         |
         | HTTP / gRPC
         v
Load Balancer (nginx / Envoy / AWS ALB)
  - SSL termination
  - Rate limiting
  - Health check routing
         |
         v
API Gateway (FastAPI / Flask)
  - Authentication / authorisation
  - Request validation
  - Usage metering
  - SSE streaming
         |
         v
Inference Engine (vLLM / TGI / TensorRT-LLM)
  - Continuous batching       (Notebook 09.02)
  - KV cache management       (Notebook 09.01)
  - Token streaming           (Notebook 09.03)
  - Quantised weights         (Notebook 09.04)
         |
         v
GPU Cluster (CUDA)
  - Tensor parallelism (multi-GPU)
  - Pipeline parallelism (multi-node)
         |
         v
Observability (Prometheus / Grafana)
  - TTFT p50/p99
  - Throughput (tokens/s)
  - GPU utilisation
  - Queue depth
  - Error rate
```

This notebook focuses on the inference engine and observability layers, which are the most LLM-specific.

In [ ]:
# Environment setup.
#
# This notebook has two tracks:
#   Track A: vLLM server setup and real benchmarking (requires GPU + vLLM)
#   Track B: serving architecture simulation, capacity planning, and
#             observability design (runs on any machine)
#
# We detect GPU availability and set flags accordingly.
# Most production insights are covered in Track B and do not require a GPU.

!pip install vllm fastapi uvicorn httpx matplotlib numpy scipy aiohttp -q 2>/dev/null || \
    pip install fastapi uvicorn httpx matplotlib numpy scipy aiohttp -q

import os
import time
import json
import asyncio
import threading
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import deque, defaultdict
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field
import warnings
warnings.filterwarnings('ignore')

import torch

HAS_GPU  = torch.cuda.is_available()
HAS_VLLM = False
try:
    import vllm
    HAS_VLLM = True
except ImportError:
    pass

print(f"GPU available: {HAS_GPU}")
print(f"vLLM available: {HAS_VLLM}")
print(f"Track: {'A (real vLLM)' if (HAS_GPU and HAS_VLLM) else 'B (simulation)'}")

## 2. vLLM Server Setup (Track A)

In [ ]:
# vLLM server configuration and startup.
#
# vLLM is the most widely used open-source LLM inference engine as of 2024.
# It implements PagedAttention (Notebook 09.01), continuous batching
# (Notebook 09.02), and streaming (Notebook 09.03) in a single package.
#
# The key vLLM configuration knobs for production:
#
#   --gpu-memory-utilization (0.0-1.0): fraction of GPU memory reserved for
#     the KV cache. The rest is used for model weights and activations.
#     Setting too high causes OOM; too low wastes capacity. Start at 0.85.
#
#   --max-model-len: maximum context window (prompt + output tokens).
#     Longer contexts use more KV cache memory per sequence.
#     Set to match your actual request distribution, not the model maximum.
#
#   --tensor-parallel-size: number of GPUs for tensor parallelism.
#     Required when the model does not fit on one GPU.
#     Use powers of 2 (1, 2, 4, 8).
#
#   --quantization: weight quantisation mode.
#     'awq', 'gptq', 'squeezellm' for pre-quantised models.
#     Set to None for full-precision inference.
#
# The server exposes an OpenAI-compatible REST API by default.
# This means any client written for the OpenAI API works without modification.

VLLM_SERVER_CONFIG = {
    'model':                   'gpt2',
    'host':                    '0.0.0.0',
    'port':                    8000,
    'gpu_memory_utilization':  0.85,
    'max_model_len':           2048,
    'tensor_parallel_size':    1,
    'dtype':                   'auto',
    'quantization':            None,
    'max_num_batched_tokens':  4096,
    'max_num_seqs':            256,
}

# The equivalent command-line invocation (for reference and documentation):
vllm_cmd = (
    f"python -m vllm.entrypoints.openai.api_server "
    f"--model {VLLM_SERVER_CONFIG['model']} "
    f"--host {VLLM_SERVER_CONFIG['host']} "
    f"--port {VLLM_SERVER_CONFIG['port']} "
    f"--gpu-memory-utilization {VLLM_SERVER_CONFIG['gpu_memory_utilization']} "
    f"--max-model-len {VLLM_SERVER_CONFIG['max_model_len']} "
    f"--max-num-batched-tokens {VLLM_SERVER_CONFIG['max_num_batched_tokens']}"
)

print("vLLM server command:")
print(f"  {vllm_cmd}")
print()

if HAS_GPU and HAS_VLLM:
    # Start vLLM in a subprocess (non-blocking)
    # The server takes 30-60 seconds to load the model before accepting requests
    print("Starting vLLM server in background... (wait 30s before benchmarking)")
    vllm_process = subprocess.Popen(vllm_cmd.split(), stdout=subprocess.DEVNULL)
    time.sleep(30)
    SERVING_URL = f"http://localhost:{VLLM_SERVER_CONFIG['port']}"
else:
    print("GPU or vLLM not available. Using simulation mode for the remainder of this notebook.")
    SERVING_URL = None

print(f"Serving URL: {SERVING_URL or '(simulation mode)'}")

## 3. Load Testing Harness

In [ ]:
# Async load testing client.
#
# A proper LLM serving benchmark must use async concurrent requests,
# not sequential requests, because sequential testing does not stress
# the batching and scheduling components -- it just measures single-request
# latency, which is the easiest case.
#
# The benchmark sends N requests concurrently, records the timestamp of:
#   - When the request was sent (request_start)
#   - When the first response chunk arrived (first_token_time)
#   - When the last response chunk arrived (last_token_time)
#   - The total number of output tokens received
#
# From these four timestamps we derive TTFT, TPOT, and E2E latency.
# We also compute server-side throughput as total_tokens / total_wall_time.
#
# The benchmark is compatible with any OpenAI-compatible API including
# vLLM, TGI, Ollama, and the real OpenAI API.

@dataclass
class BenchmarkRequest:
    prompt:          str
    max_tokens:      int = 200


@dataclass
class BenchmarkResult:
    request_id:       int
    prompt_tokens:    int
    output_tokens:    int
    request_start:    float
    first_token_time: Optional[float] = None
    last_token_time:  Optional[float] = None
    error:            Optional[str]   = None

    @property
    def ttft_ms(self) -> Optional[float]:
        if self.first_token_time:
            return (self.first_token_time - self.request_start) * 1000
        return None

    @property
    def e2e_ms(self) -> Optional[float]:
        if self.last_token_time:
            return (self.last_token_time - self.request_start) * 1000
        return None

    @property
    def tpot_ms(self) -> Optional[float]:
        """Time per output token after the first (ms)."""
        if self.first_token_time and self.last_token_time and self.output_tokens > 1:
            return (self.last_token_time - self.first_token_time) * 1000 / (self.output_tokens - 1)
        return None


async def send_streaming_request(
    session,
    base_url:  str,
    req:       BenchmarkRequest,
    req_id:    int,
) -> BenchmarkResult:
    """
    Send one request to an OpenAI-compatible streaming endpoint.

    Parses the SSE stream and records timestamps for TTFT measurement.
    Compatible with /v1/completions (legacy) and /v1/chat/completions.
    """
    payload = {
        'model':       'gpt2',
        'prompt':      req.prompt,
        'max_tokens':  req.max_tokens,
        'stream':      True,
        'temperature': 1.0,
    }

    result = BenchmarkResult(
        request_id=req_id,
        prompt_tokens=len(req.prompt.split()),  # approximate
        output_tokens=0,
        request_start=time.time(),
    )

    try:
        import aiohttp
        async with session.post(
            f"{base_url}/v1/completions",
            json=payload,
            timeout=aiohttp.ClientTimeout(total=120),
        ) as response:
            async for line in response.content:
                line = line.decode('utf-8').strip()
                if not line.startswith('data: '):
                    continue
                data_str = line[6:]
                if data_str == '[DONE]':
                    result.last_token_time = time.time()
                    break
                try:
                    chunk = json.loads(data_str)
                    token_text = chunk['choices'][0].get('text', '')
                    if token_text:
                        if result.first_token_time is None:
                            result.first_token_time = time.time()
                        result.output_tokens += 1
                except (json.JSONDecodeError, KeyError):
                    pass
    except Exception as e:
        result.error = str(e)

    if result.last_token_time is None:
        result.last_token_time = time.time()

    return result


async def run_benchmark(
    base_url:        str,
    requests:        List[BenchmarkRequest],
    concurrency:     int = 10,
) -> List[BenchmarkResult]:
    """
    Run concurrent benchmark requests against an LLM serving endpoint.

    Uses an asyncio semaphore to limit concurrency to the specified level.
    Each request records TTFT, TPOT, and E2E latency independently.
    """
    import aiohttp
    semaphore = asyncio.Semaphore(concurrency)
    results   = []

    async def bounded_request(session, req, req_id):
        async with semaphore:
            return await send_streaming_request(session, base_url, req, req_id)

    async with aiohttp.ClientSession() as session:
        tasks = [
            bounded_request(session, req, i)
            for i, req in enumerate(requests)
        ]
        results = await asyncio.gather(*tasks)

    return list(results)


print("Load testing harness defined.")
print()

# Sample benchmark requests
BENCHMARK_PROMPTS = [
    "Explain the key differences between supervised and unsupervised learning.",
    "What are the main components of a transformer architecture?",
    "Describe the attention mechanism in three sentences.",
    "What is gradient descent and why is it used in neural networks?",
    "Explain what regularisation is and give two examples.",
    "How does batch normalisation improve training stability?",
    "What is the vanishing gradient problem and how can it be addressed?",
    "Describe the role of residual connections in deep networks.",
]

BENCHMARK_REQUESTS = [
    BenchmarkRequest(prompt=p, max_tokens=150)
    for p in BENCHMARK_PROMPTS * 4   # 32 total requests
]

print(f"Benchmark: {len(BENCHMARK_REQUESTS)} requests, {len(BENCHMARK_PROMPTS)} unique prompts")

if SERVING_URL:
    print(f"Running against: {SERVING_URL}")
    print("(Set SERVING_URL to None or leave GPU unavailable to use simulation mode)")

In [ ]:
# Run the benchmark (or simulate results if no server is available).
#
# When a real vLLM server is running, this cell sends actual requests and
# records real timing data. When no server is available, we generate
# realistic simulated results using distributions from the serving literature.
#
# The simulated data is calibrated to realistic production values:
#   TTFT: log-normal, mean ~150ms (prefill latency for moderate prompt length)
#   TPOT: log-normal, mean ~25ms (40 tokens/s on a single A100)
#   Output tokens: log-normal, mean ~120 tokens (typical response length)
#
# These values vary significantly depending on:
#   - Model size (7B vs 70B)
#   - GPU type (A10G vs A100 vs H100)
#   - Quantisation (fp16 vs int8 vs nf4)
#   - Concurrent load (higher load = higher TTFT from queue wait)

def simulate_benchmark_results(
    n_requests: int = 32,
    concurrency: int = 8,
    ttft_mean_ms: float = 150.0,
    tpot_mean_ms: float = 25.0,
    output_tokens_mean: int = 120,
) -> List[BenchmarkResult]:
    """
    Generate realistic simulated benchmark results.

    Concurrency increases TTFT due to queue waiting time.
    The multiplier scales TTFT linearly with load -- a simplification
    that holds well below the saturation point.
    """
    rng = np.random.RandomState(42)
    load_factor = 1 + (concurrency - 1) * 0.15   # TTFT grows with load

    results = []
    t_start = time.time()

    for i in range(n_requests):
        request_start = t_start + i * 0.05  # stagger arrivals slightly

        ttft_s  = rng.lognormal(
            np.log(ttft_mean_ms * load_factor / 1000),
            0.35
        )
        n_out   = max(10, int(rng.lognormal(np.log(output_tokens_mean), 0.5)))
        tpot_s  = rng.lognormal(np.log(tpot_mean_ms / 1000), 0.25)
        total_s = ttft_s + (n_out - 1) * tpot_s

        results.append(BenchmarkResult(
            request_id=i,
            prompt_tokens=rng.randint(20, 60),
            output_tokens=n_out,
            request_start=request_start,
            first_token_time=request_start + ttft_s,
            last_token_time=request_start + total_s,
        ))

    return results


# Run real or simulated benchmark
if SERVING_URL:
    print("Running real benchmark against vLLM server...")
    benchmark_results = asyncio.run(run_benchmark(
        SERVING_URL, BENCHMARK_REQUESTS, concurrency=8
    ))
else:
    print("Running simulated benchmark (realistic production values)...")
    benchmark_results = simulate_benchmark_results(
        n_requests=32, concurrency=8
    )

# Compute summary statistics
successful = [r for r in benchmark_results if r.error is None]
ttft_ms    = [r.ttft_ms  for r in successful if r.ttft_ms  is not None]
tpot_ms    = [r.tpot_ms  for r in successful if r.tpot_ms  is not None]
e2e_ms     = [r.e2e_ms   for r in successful if r.e2e_ms   is not None]
out_tokens = [r.output_tokens for r in successful]

total_wall = max(r.last_token_time for r in successful) - min(r.request_start for r in successful)
total_toks = sum(out_tokens)
throughput_tps = total_toks / total_wall

print(f"\nBenchmark results ({len(successful)}/{len(benchmark_results)} requests succeeded):")
print()
print(f"{'Metric':<30} {'p50':>8} {'p95':>8} {'p99':>8} {'mean':>8}")
print("-" * 58)
for name, data in [
    ('TTFT (ms)',              ttft_ms),
    ('TPOT (ms/token)',        tpot_ms),
    ('E2E latency (ms)',       e2e_ms),
    ('Output tokens',          out_tokens),
]:
    if data:
        p50 = np.percentile(data, 50)
        p95 = np.percentile(data, 95)
        p99 = np.percentile(data, 99)
        mean = np.mean(data)
        print(f"{name:<30} {p50:>8.1f} {p95:>8.1f} {p99:>8.1f} {mean:>8.1f}")
print()
print(f"Total output tokens: {total_toks:,}")
print(f"Wall-clock time:     {total_wall:.1f}s")
print(f"Throughput:          {throughput_tps:.0f} tokens/s")

In [ ]:
# Benchmark visualisation dashboard.
#
# The four panels cover the key production serving metrics:
#
# Top-left: TTFT CDF. The x-axis value at y=0.95 is your p95 TTFT SLA.
#   For a conversational assistant, a p95 TTFT under 500ms is generally
#   considered acceptable. Above 1000ms users begin to notice the wait.
#
# Top-right: TPOT distribution. This governs the streaming speed.
#   At 25ms/token the user sees ~40 tokens/s, which is faster than most
#   people can read. At 100ms/token it starts to feel noticeably slow.
#
# Bottom-left: E2E latency by output length. Shows whether latency
#   scales linearly with output length as expected (it should).
#   Non-linear scaling would indicate batching stalls or preemptions.
#
# Bottom-right: Concurrency sweep. Shows how TTFT and throughput change
#   as we increase the number of concurrent requests. This identifies
#   the optimal operating point: maximum throughput before TTFT degrades.

fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)

# TTFT CDF
ax1 = fig.add_subplot(gs[0, 0])
ttft_sorted = sorted(ttft_ms)
cdf = np.arange(1, len(ttft_sorted)+1) / len(ttft_sorted)
ax1.plot(ttft_sorted, cdf, color='#1f77b4', linewidth=2.5)
for pct, color, label in [(50, '#2ca02c', 'p50'), (95, '#ff7f0e', 'p95'), (99, '#d62728', 'p99')]:
    val = np.percentile(ttft_ms, pct)
    ax1.axvline(val, color=color, linestyle='--', alpha=0.7, label=f'{label}={val:.0f}ms')
ax1.set_xlabel('Time to First Token (ms)', fontsize=11)
ax1.set_ylabel('CDF', fontsize=11)
ax1.set_title('TTFT Cumulative Distribution\n'
              'p95 is the key SLA metric for user experience', fontsize=12)
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# TPOT distribution
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(tpot_ms, bins=25, color='steelblue', alpha=0.8, edgecolor='white')
ax2.axvline(np.mean(tpot_ms), color='red', linestyle='--',
            label=f'Mean = {np.mean(tpot_ms):.1f} ms ({1000/np.mean(tpot_ms):.0f} t/s)')
ax2.axvline(25, color='green', linestyle=':', alpha=0.7, label='Target: 25ms (40 t/s)')
ax2.set_xlabel('Time per Output Token (ms)', fontsize=11)
ax2.set_ylabel('Count', fontsize=11)
ax2.set_title('TPOT Distribution\n'
              'Governs perceived streaming speed after TTFT', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

# E2E latency vs output tokens scatter
ax3 = fig.add_subplot(gs[1, 0])
scatter_tokens = [r.output_tokens for r in successful]
scatter_e2e    = [r.e2e_ms for r in successful]
ax3.scatter(scatter_tokens, scatter_e2e, alpha=0.5, color='#1f77b4', s=30)
# Fit a line to check linearity
if len(scatter_tokens) > 5:
    z = np.polyfit(scatter_tokens, scatter_e2e, 1)
    p = np.poly1d(z)
    x_fit = np.linspace(min(scatter_tokens), max(scatter_tokens), 100)
    ax3.plot(x_fit, p(x_fit), 'r--', alpha=0.7,
             label=f'Linear fit: {z[0]:.1f}ms/token')
ax3.set_xlabel('Output tokens', fontsize=11)
ax3.set_ylabel('E2E latency (ms)', fontsize=11)
ax3.set_title('E2E Latency vs Response Length\n'
              'Should be linear -- non-linearity indicates batching stalls', fontsize=12)
ax3.legend(fontsize=9)
ax3.grid(alpha=0.3)

# Concurrency sweep
ax4 = fig.add_subplot(gs[1, 1])
concurrencies   = [1, 2, 4, 8, 16, 32]
sweep_ttft_p95  = []
sweep_throughput = []
for conc in concurrencies:
    sim = simulate_benchmark_results(n_requests=conc*4, concurrency=conc)
    succ = [r for r in sim if r.error is None]
    ttfts = [r.ttft_ms for r in succ if r.ttft_ms]
    wall  = max(r.last_token_time for r in succ) - min(r.request_start for r in succ)
    toks  = sum(r.output_tokens for r in succ)
    sweep_ttft_p95.append(np.percentile(ttfts, 95))
    sweep_throughput.append(toks / wall)

ax4_twin = ax4.twinx()
line1, = ax4.plot(concurrencies, sweep_ttft_p95, 'o-', color='#d62728', linewidth=2, label='p95 TTFT (ms)')
line2, = ax4_twin.plot(concurrencies, sweep_throughput, 's-', color='#2ca02c', linewidth=2, label='Throughput (t/s)')
ax4.set_xlabel('Concurrent requests', fontsize=11)
ax4.set_ylabel('p95 TTFT (ms)', color='#d62728', fontsize=11)
ax4_twin.set_ylabel('Throughput (tokens/s)', color='#2ca02c', fontsize=11)
ax4.set_title('Concurrency Sweep\n'
              'Find the "knee": max throughput before TTFT degrades', fontsize=12)
lines = [line1, line2]
ax4.legend(lines, [l.get_label() for l in lines], fontsize=9)
ax4.grid(alpha=0.3)

plt.suptitle('LLM Serving Benchmark Dashboard', fontsize=14, fontweight='bold')
plt.savefig('../figures/09_serving_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Capacity Planning

In [ ]:
# GPU capacity planning formula.
#
# Given a traffic forecast and SLA requirements, how many GPUs do you need?
#
# The fundamental constraint is tokens-per-second per GPU. Every other
# variable -- number of users, request rate, response length -- must
# eventually be reduced to total token throughput.
#
# Model memory is the first constraint: model weights must fit on each GPU.
# KV cache is the second: the remaining memory determines the maximum
# batch size and therefore the achievable throughput.
#
# The formula below is a first-order estimate. Real capacity requires
# load testing your specific model + hardware combination.
#
# Reference GPU throughputs (fp16, approximate):
#   A10G (24 GB VRAM):  Llama-2-7B  ~2,000 t/s  at max batch
#   A100 (80 GB VRAM):  Llama-2-13B ~3,500 t/s  at max batch
#   H100 (80 GB VRAM):  Llama-2-70B ~2,800 t/s  at max batch

@dataclass
class CapacityPlan:
    model_name:           str
    model_size_b:         float    # parameter count in billions
    precision_bytes:      int      # bytes per parameter (2=fp16, 1=int8, 0.5=int4)
    gpu_vram_gb:          float    # GPU VRAM in GB
    gpu_tokens_per_sec:   float    # measured or estimated throughput
    daily_active_users:   int
    requests_per_user:    float    # mean requests per user per day
    mean_output_tokens:   int      # mean output tokens per request
    peak_to_mean_ratio:   float    # how much peak traffic exceeds mean
    sla_ttft_p95_ms:      float    # SLA target for p95 TTFT

    @property
    def model_size_gb(self) -> float:
        return self.model_size_b * 1e9 * self.precision_bytes / 1e9

    @property
    def gpus_for_one_instance(self) -> int:
        """Minimum GPUs to fit the model (tensor parallelism)."""
        return max(1, int(np.ceil(self.model_size_gb / self.gpu_vram_gb)))

    @property
    def mean_tokens_per_second(self) -> float:
        """Mean token throughput demand."""
        mean_rps = self.daily_active_users * self.requests_per_user / 86400
        return mean_rps * self.mean_output_tokens

    @property
    def peak_tokens_per_second(self) -> float:
        return self.mean_tokens_per_second * self.peak_to_mean_ratio

    @property
    def n_instances_required(self) -> int:
        """Number of serving instances needed to handle peak traffic."""
        instances = self.peak_tokens_per_second / self.gpu_tokens_per_sec
        return max(1, int(np.ceil(instances * 1.25)))  # 25% headroom

    @property
    def total_gpus_required(self) -> int:
        return self.n_instances_required * self.gpus_for_one_instance

    def print_plan(self):
        print(f"Capacity plan: {self.model_name}")
        print(f"  Model size:         {self.model_size_gb:.1f} GB  "
              f"(requires {self.gpus_for_one_instance} GPU/instance for tensor parallelism)")
        print(f"  Mean traffic:       {self.mean_tokens_per_second:.0f} tokens/s")
        print(f"  Peak traffic:       {self.peak_tokens_per_second:.0f} tokens/s "
              f"(peak/mean = {self.peak_to_mean_ratio:.1f}x)")
        print(f"  Per-GPU throughput: {self.gpu_tokens_per_sec:.0f} tokens/s")
        print(f"  Instances needed:   {self.n_instances_required} "
              f"(incl. 25% headroom)")
        print(f"  Total GPUs needed:  {self.total_gpus_required}")


# Example capacity plans
scenarios = [
    CapacityPlan(
        model_name='Llama-3-8B on A10G',
        model_size_b=8, precision_bytes=2, gpu_vram_gb=24,
        gpu_tokens_per_sec=2000,
        daily_active_users=10_000, requests_per_user=5, mean_output_tokens=200,
        peak_to_mean_ratio=3.0, sla_ttft_p95_ms=500,
    ),
    CapacityPlan(
        model_name='Llama-3-70B on A100',
        model_size_b=70, precision_bytes=2, gpu_vram_gb=80,
        gpu_tokens_per_sec=2800,
        daily_active_users=10_000, requests_per_user=5, mean_output_tokens=200,
        peak_to_mean_ratio=3.0, sla_ttft_p95_ms=500,
    ),
    CapacityPlan(
        model_name='Llama-3-70B NF4 on A100',
        model_size_b=70, precision_bytes=0.5, gpu_vram_gb=80,
        gpu_tokens_per_sec=3500,   # NF4 faster due to larger batch in same KV budget
        daily_active_users=10_000, requests_per_user=5, mean_output_tokens=200,
        peak_to_mean_ratio=3.0, sla_ttft_p95_ms=500,
    ),
]

for plan in scenarios:
    print()
    plan.print_plan()

print()
print("Observation: quantisation (NF4) reduces GPU requirements by allowing")
print("70B to fit on a single A100 and increasing effective throughput via")
print("larger batch sizes in the same KV cache budget.")

In [ ]:
# Production observability setup.
#
# Every production LLM serving deployment needs four categories of metrics:
#
#   1. Latency: TTFT, TPOT, and E2E latency at p50/p95/p99
#   2. Throughput: requests/s, tokens/s, and utilisation
#   3. Quality: error rate, timeout rate, empty response rate
#   4. Resource: GPU memory, GPU utilisation, queue depth
#
# We implement a minimal Prometheus-compatible metrics collector.
# In production, metrics are scraped by Prometheus and visualised in Grafana.
# vLLM exposes all four categories of metrics at /metrics out of the box.
#
# The key alerting rules for an LLM serving deployment:
#   - TTFT p95 > SLA threshold -> scale up or investigate bottleneck
#   - GPU memory > 90% -> risk of OOM, reduce batch size or add GPUs
#   - Queue depth > 50 -> requests are waiting longer than expected
#   - Error rate > 1% -> investigate model crashes or upstream issues

class ServingMetricsCollector:
    """
    Lightweight metrics collector for LLM serving observability.

    Maintains a sliding window of recent requests for real-time percentile
    computation without storing the full history.
    """

    def __init__(self, window_size: int = 1000):
        self.window_size   = window_size
        self.ttft_ms       = deque(maxlen=window_size)
        self.tpot_ms       = deque(maxlen=window_size)
        self.e2e_ms        = deque(maxlen=window_size)
        self.output_tokens = deque(maxlen=window_size)
        self.errors        = deque(maxlen=window_size)
        self.total_requests = 0
        self.total_tokens   = 0
        self.start_time     = time.time()

    def record(self, result: BenchmarkResult):
        """Record metrics from a completed request."""
        self.total_requests += 1
        if result.error:
            self.errors.append(1)
            return
        self.errors.append(0)
        if result.ttft_ms:  self.ttft_ms.append(result.ttft_ms)
        if result.tpot_ms:  self.tpot_ms.append(result.tpot_ms)
        if result.e2e_ms:   self.e2e_ms.append(result.e2e_ms)
        self.output_tokens.append(result.output_tokens)
        self.total_tokens += result.output_tokens

    def summary(self) -> Dict:
        """Return current metric values as a dict suitable for Prometheus export."""
        elapsed = time.time() - self.start_time

        def percentiles(data, ps=(50, 95, 99)):
            if not data: return {p: None for p in ps}
            arr = np.array(data)
            return {p: np.percentile(arr, p) for p in ps}

        return {
            'total_requests':   self.total_requests,
            'error_rate':       np.mean(self.errors) if self.errors else 0.0,
            'throughput_rps':   self.total_requests / elapsed,
            'throughput_tps':   self.total_tokens / elapsed,
            'ttft_ms':          percentiles(self.ttft_ms),
            'tpot_ms':          percentiles(self.tpot_ms),
            'e2e_ms':           percentiles(self.e2e_ms),
            'mean_output_tokens': np.mean(self.output_tokens) if self.output_tokens else 0.0,
        }

    def print_dashboard(self):
        s = self.summary()
        print("Serving Metrics Dashboard")
        print(f"  Requests:    {s['total_requests']:>8,}  "
              f"Error rate: {s['error_rate']:>6.1%}  "
              f"Throughput: {s['throughput_tps']:>7.0f} t/s")
        print(f"  TTFT (ms):   p50={s['ttft_ms'][50]:>6.0f}  "
              f"p95={s['ttft_ms'][95]:>6.0f}  p99={s['ttft_ms'][99]:>6.0f}")
        print(f"  TPOT (ms):   p50={s['tpot_ms'][50]:>6.1f}  "
              f"p95={s['tpot_ms'][95]:>6.1f}  p99={s['tpot_ms'][99]:>6.1f}")
        print(f"  E2E (ms):    p50={s['e2e_ms'][50]:>6.0f}  "
              f"p95={s['e2e_ms'][95]:>6.0f}  p99={s['e2e_ms'][99]:>6.0f}")


# Feed the benchmark results into the collector
collector = ServingMetricsCollector()
for r in benchmark_results:
    collector.record(r)

print()
collector.print_dashboard()
print()
print("In production, this dashboard updates every 30 seconds.")
print("Alerts fire when p95 TTFT > SLA threshold or error rate > 1%.")

## 5. Engineering Notes

**Production LLM serving deployment checklist**

| Category | Item | Notes |
|----------|------|-------|
| **Inference engine** | Enable continuous batching | vLLM default; do not use naive HuggingFace generate() |
| | Set `gpu_memory_utilization=0.85` | Leave 15% for CUDA kernels and activations |
| | Enable prefix caching | Saves KV compute on shared system prompts |
| | Use bf16 (not fp32) | Halves memory, ~identical quality |
| **Latency** | Monitor TTFT separately from E2E | TTFT is what users feel; E2E includes generation time |
| | Set request timeout at gateway | Cancel requests that have waited > 2x SLA target |
| | Enable chunked prefill | Reduces TTFT variance for long-prompt requests |
| **Throughput** | Tune `max_num_batched_tokens` | Start at 4096; increase until GPU memory is the limit |
| | Use tensor parallelism for 70B+ models | Spread across 4+ GPUs |
| | Consider NF4 for 3.5x memory reduction | Throughput often increases due to larger effective batch |
| **Observability** | Export TTFT p50/p95/p99 to Prometheus | Use vLLM's built-in `/metrics` endpoint |
| | Alert on GPU memory > 90% | Risk of OOM-induced crashes |
| | Alert on queue depth > 50 | Indicates under-provisioning |
| **Reliability** | Run health checks every 10s | Restart hung workers automatically |
| | Use rolling restarts for updates | Zero-downtime model version upgrades |
| | Set max_seq_len to match workload | Oversized context window wastes KV cache |

## 6. Module 09 Summary

This module covered the complete LLM inference engineering stack:

- **09.01**: KV Cache -- why it is fundamental and how to compute memory requirements
- **09.02**: Dynamic Batching -- continuous batching doubles throughput over static batching
- **09.03**: Token Streaming -- SSE protocol, backpressure, and perceived latency
- **09.04**: Quantisation -- INT8/NF4 from scratch; 4x memory reduction with ~3% accuracy cost
- **09.05**: LLM Serving -- vLLM setup, load testing, capacity planning, observability

## 7. Exercises

1. **End-to-end vLLM benchmark**: Set up vLLM with GPT-2, run the full benchmark harness, and compare real measurements to the simulated values. Where do the distributions differ most?

2. **Capacity plan for your use case**: Use the `CapacityPlan` class with your own traffic estimates (daily active users, requests per user, mean output length). How many A10G GPUs would you need for 10,000 DAU with 5 requests/day?

3. **Prometheus integration**: Extend the `ServingMetricsCollector` to expose a `/metrics` endpoint in Prometheus text format. Test with `curl` and verify the output can be scraped.

4. **Concurrency sweep**: Use the benchmark harness to sweep concurrency from 1 to 64 against a local vLLM server. Plot TTFT p95 and throughput. At what concurrency does TTFT start to degrade?

5. **Quantisation in production**: Load the same model with `load_in_4bit=True` and without quantisation. Run both through the benchmark harness at concurrency=16. What is the throughput improvement from quantisation? Does p95 TTFT improve or worsen?

---

## 8. References

- [Kwon et al. (2023) -- Efficient Memory Management for Large Language Model Serving with PagedAttention](https://arxiv.org/abs/2309.06180)
- [vLLM Documentation](https://docs.vllm.ai)
- [TGI (Text Generation Inference) Documentation](https://huggingface.co/docs/text-generation-inference)
- [Agrawal et al. (2024) -- Taming Throughput-Latency Tradeoff in LLM Inference](https://arxiv.org/abs/2407.11418)
- [Sheng et al. (2023) -- FlexGen: High-Throughput Generative Inference of Large Language Models with a Single GPU](https://arxiv.org/abs/2303.06865)